In [1]:
!pip install dash

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/7.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/7.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.8/7.2 MB 2.0 MB/s eta 0:00:04
   ---- ----------------------------------- 0.8/7.2 MB 2.0 MB/s eta 0:00:04
   ------- -------------------------------- 1.3/7.2 MB 1.6 MB/s eta 0:00:04
   ---------- ----------------------------- 1.8/7.2 MB 1.6 MB/s eta 0:00:04
   ------------- -------------------------- 2.4/7.2 MB 1.8 MB/s eta 0:00:03
   --------------- ------------------------ 2.9/7.2 MB 1.9 MB/s eta 0:00:03
   ------------------ --------------------- 3.4/7.2 MB 2.0 MB/s eta 0:00:02
   --------------------- ------------------ 3.9/7.2 MB 2.1 MB/s eta 0:00:02
   ------------------------ --------------- 4.5/7.2 MB 2.1 MB/s eta 0:00:02
   --------------------------- 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import dash
from dash import dcc, html, Input, Output
import pandas as pd
import plotly.express as px

# 1. Load and Prepare the Data (Same as Activity 5)
df = pd.read_csv('Coffe_sales.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Group by Week to make the data smooth and clear
weekly_sales = df.groupby([pd.Grouper(key='Date', freq='W'), 'coffee_name'])['money'].sum().reset_index()

# Get a list of unique coffee names for our dropdown menu
coffee_options = weekly_sales['coffee_name'].unique().tolist()

# 2. Initialize the Dash App
app = dash.Dash(__name__)

# 3. Define the App Layout (The User Interface)
app.layout = html.Div([
    html.H1("Coffee Sales Dashboard", style={'textAlign': 'center', 'fontFamily': 'Arial'}),
    
    html.Label("Select Coffee Type(s):", style={'fontFamily': 'Arial', 'fontWeight': 'bold'}),
    
    # Add the Interactive Dropdown
    dcc.Dropdown(
        id='coffee-dropdown',
        options=[{'label': coffee, 'value': coffee} for coffee in coffee_options],
        value=coffee_options, # By default, all coffees are selected
        multi=True,           # Allow selecting multiple coffees
        style={'marginBottom': '20px'}
    ),
    
    # Placeholder for our Plotly Graph
    dcc.Graph(id='coffee-graph')
])

# 4. Add the Callback (The Interactive Logic)
# This links the Dropdown (Input) to the Graph (Output)
@app.callback(
    Output('coffee-graph', 'figure'),
    Input('coffee-dropdown', 'value')
)
def update_graph(selected_coffees):
    # Filter the dataset based on what the user selected in the dropdown
    # If nothing is selected, return an empty figure to prevent errors
    if not selected_coffees:
        return px.line(title="Please select at least one coffee type.")
        
    filtered_df = weekly_sales[weekly_sales['coffee_name'].isin(selected_coffees)]
    
    # Generate the Plotly chart with the filtered data
    fig = px.line(
        filtered_df,
        x='Date',
        y='money',
        color='coffee_name',
        markers=True,
        title='Weekly Coffee Revenue',
        labels={'money': 'Revenue ($)', 'Date': 'Date', 'coffee_name': 'Coffee Type'},
        template='plotly_white'
    )
    
    # Add the unified hover mode we used in Activity 5
    fig.update_layout(hovermode="x unified")
    return fig

# 5. Run the App
if __name__ == '__main__':
    # Runs the app on a local server
    app.run(jupyter_mode="external", debug=True, port=8050)

Dash app running on http://127.0.0.1:8050/
